In [ ]:
"""
=====================================================================
TCC - Extensão Histórica da Série de Produção Cafeeira via SIDRA/IBGE
=====================================================================

Este script busca, diretamente da API pública do IBGE, os dados de
produção de café para os municípios da microrregião de São Sebastião
do Paraíso, cobrindo o período histórico.

Correções implementadas nesta versão:
  1. Busca por "Café (em grão) Total" (o IBGE não separava Arábica nos anos 80/90).
  2. Leitura dinâmica do cabeçalho para descobrir exatamente onde está
     o Ano (D2N, D3N, etc), evitando erros de conversão int().
  3. Tratamento de caracteres nulos do IBGE ("-", "..", "X").
"""

import requests
import pandas as pd
import time
import numpy as np

# =====================================================================
# CONFIGURAÇÃO
# =====================================================================
MUNICIPIOS_ALVO = [
    "Arceburgo", "Cabo Verde", "Guaranésia", "Guaxupé", "Itamogi",
    "Jacuí", "Juruaia", "Monte Belo", "Muzambinho", "Nova Resende"
]
UF = "MG"

TABELA_SIDRA = "1613"          # Lavouras permanentes, série 1974-2023
ANO_INICIO = 1988              # ano inicial da extensão
ANO_FIM = 2012                 # até o ano anterior ao início da base atual (2013)

NOMES_VARIAVEIS_DESEJADAS = {
    "area_colhida": "Área colhida",
    "quantidade_produzida": "Quantidade produzida",
    "rendimento_medio": "Rendimento médio da produção",
}

# CORREÇÃO HISTÓRICA: Antes dos anos 2000, não existia a quebra por "Arábica".
# Tem que ser o Total para não vir vazio.
PRODUTO_DESEJADO = "café (em grão) Total"

ARQUIVO_SAIDA = "TCC_SIDRA_Historico_1988_2012.csv"

# =====================================================================
# 1. RESOLVER CÓDIGOS IBGE DOS MUNICÍPIOS AUTOMATICAMENTE
# =====================================================================
def obter_codigos_municipios(municipios, uf):
    print(f"[INFO] Buscando códigos IBGE para {len(municipios)} municípios de {uf}...")
    url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{uf}/municipios"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    todos = resp.json()

    def normalizar(s):
        import unicodedata
        s = unicodedata.normalize('NFKD', s).encode('ASCII', 'ignore').decode('ASCII')
        return s.lower().strip()

    mapa_nomes = {normalizar(m['nome']): m['id'] for m in todos}

    codigos = {}
    faltantes = []
    for nome in municipios:
        chave = normalizar(nome)
        if chave in mapa_nomes:
            codigos[nome] = mapa_nomes[chave]
        else:
            faltantes.append(nome)

    if faltantes:
        print(f"[AVISO] Não foi possível resolver automaticamente: {faltantes}")

    print(f"[OK] {len(codigos)} códigos resolvidos.")
    return codigos

# =====================================================================
# 2. DESCOBRIR O CÓDIGO DE CLASSIFICAÇÃO DO CAFÉ NA TABELA 1613
# =====================================================================
def obter_codigo_classificacao_cafe(tabela, produto_desejado):
    print(f"\n[INFO] Consultando descritor da Tabela {tabela} para localizar '{produto_desejado}'...")
    url = f"https://apisidra.ibge.gov.br/DescritoresTabela/t/{tabela}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    descritor = resp.json()

    classificacoes = descritor.get("classificacoes", descritor.get("Classificacoes", []))

    for c in classificacoes:
        categorias = c.get("categorias", c.get("Categorias", []))
        for cat in categorias:
            nome_cat = cat.get("nome", cat.get("Nome", ""))
            if produto_desejado.lower() in nome_cat.lower():
                codigo_classificacao = c.get("id", c.get("Id"))
                codigo_categoria = cat.get("id", cat.get("Id"))
                print(f"[OK] Encontrado: classificação={codigo_classificacao}, categoria={codigo_categoria} ('{nome_cat}')")
                return codigo_classificacao, codigo_categoria

    print("[ERRO] Produto não encontrado.")
    return None, None

# =====================================================================
# 3. DESCOBRIR OS CÓDIGOS DAS VARIÁVEIS DESEJADAS
# =====================================================================
def obter_codigos_variaveis(tabela, nomes_desejados):
    url = f"https://apisidra.ibge.gov.br/DescritoresTabela/t/{tabela}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    descritor = resp.json()

    variaveis = descritor.get("variaveis", descritor.get("Variaveis", []))
    codigos = {}
    for chave, nome_busca in nomes_desejados.items():
        for v in variaveis:
            nome_v = v.get("nome", v.get("Nome", ""))
            if nome_busca.lower() in nome_v.lower():
                codigos[chave] = v.get("id", v.get("Id"))
                break

    print(f"[OK] Códigos de variáveis resolvidos: {codigos}")
    return codigos

# =====================================================================
# 4. BUSCAR OS DADOS PARA CADA MUNICÍPIO (CORRIGIDO)
# =====================================================================
def buscar_dados_municipio(codigo_municipio, nome_municipio, tabela,
                           codigo_var, codigo_classificacao, codigo_categoria,
                           ano_inicio, ano_fim):
    periodo = f"{ano_inicio}-{ano_fim}"
    url = (
        f"https://apisidra.ibge.gov.br/values/"
        f"t/{tabela}/n6/{codigo_municipio}/p/{periodo}/"
        f"v/{codigo_var}/c{codigo_classificacao}/{codigo_categoria}"
    )

    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        dados_brutos = resp.json()

        # A linha 0 é o cabeçalho. Vamos usá-lo para descobrir onde está o 'Ano'
        cabecalho = dados_brutos[0]
        dados = dados_brutos[1:]

        # Descobre qual "D1N", "D2N", "D3N" corresponde à palavra "Ano"
        chave_ano = None
        for chave, valor in cabecalho.items():
            if valor == "Ano" or valor == "Ano (Código)":
                # Queremos sempre a chave com 'N' no final (Nome/Valor real)
                chave_ano = chave.replace('C', 'N') if 'C' in chave else chave
                break

        # Fallback de segurança se não achar a palavra Ano exata
        if not chave_ano:
            chave_ano = "D2N"

        registros = []
        for d in dados:
            # Pega o ano da coluna correta
            ano_str = str(d.get(chave_ano, ""))

            # Limpa o valor (O IBGE usa "-", "..", "X" para dados vazios)
            valor_str = str(d.get("V", ""))
            if valor_str in ["-", "..", "...", "X", "x", ""]:
                valor_numerico = np.nan
            else:
                try:
                    valor_numerico = float(valor_str)
                except ValueError:
                    valor_numerico = np.nan

            # Só adiciona se o ano for válido
            if ano_str.isdigit():
                registros.append({
                    "Municipio": nome_municipio,
                    "Ano": int(ano_str),
                    "Valor": valor_numerico
                })

        return registros

    except Exception as e:
        print(f"[ERRO] Falha ao buscar dados de {nome_municipio}: {e}")
        return []

# =====================================================================
# EXECUÇÃO PRINCIPAL
# =====================================================================
if __name__ == "__main__":
    codigos_municipios = obter_codigos_municipios(MUNICIPIOS_ALVO, UF)
    codigo_classificacao, codigo_categoria = obter_codigo_classificacao_cafe(
        TABELA_SIDRA, PRODUTO_DESEJADO
    )
    codigos_variaveis = obter_codigos_variaveis(TABELA_SIDRA, NOMES_VARIAVEIS_DESEJADAS)

    if not codigo_classificacao or not codigos_variaveis:
        print("\n[ERRO FATAL] Não foi possível resolver os parâmetros da API.")
        exit(1)

    todos_registros = {"area_colhida": [], "quantidade_produzida": [], "rendimento_medio": []}

    for nome_mun, codigo_mun in codigos_municipios.items():
        print(f"\n[INFO] Buscando dados de {nome_mun} ({codigo_mun})...")
        for chave, codigo_var in codigos_variaveis.items():
            registros = buscar_dados_municipio(
                codigo_mun, nome_mun, TABELA_SIDRA, codigo_var,
                codigo_classificacao, codigo_categoria, ANO_INICIO, ANO_FIM
            )
            todos_registros[chave].extend(registros)
            time.sleep(0.3)  # Evita bloqueio do servidor do IBGE

    # Monta os DataFrames de cada variável e faz merge
    dfs = {}
    for chave, registros in todos_registros.items():
        if registros:
            df = pd.DataFrame(registros).rename(columns={"Valor": chave})
            dfs[chave] = df

    if not dfs:
        print("\n[ERRO] Nenhum dado retornado pela API. A tabela veio vazia.")
        exit(1)

    df_final = None
    for chave, df in dfs.items():
        if df_final is None:
            df_final = df
        else:
            df_final = df_final.merge(df, on=["Municipio", "Ano"], how="outer")

    # Renomeia para o padrão do TCC
    df_final = df_final.rename(columns={
        "area_colhida": "Area_Colhida_ha",
        "quantidade_produzida": "Producao_t",
        "rendimento_medio": "Rendimento_kg_ha",
    })

    df_final = df_final.dropna(subset=['Ano'])
    df_final['Ano'] = df_final['Ano'].astype(int)
    df_final = df_final.sort_values(by=["Municipio", "Ano"]).reset_index(drop=True)

    df_final.to_csv(ARQUIVO_SAIDA, index=False)

    print(f"\n==================================================")
    print(f"[SUCESSO] Dados salvos em: {ARQUIVO_SAIDA}")
    print(f"Linhas obtidas: {len(df_final)}")
    print(f"==================================================")
    print(df_final.head(10).to_string(index=False))

[INFO] Buscando códigos IBGE para 10 municípios de MG...
[OK] 10 códigos resolvidos.

[INFO] Consultando descritor da Tabela 1613 para localizar 'café (em grão) Total'...
[OK] Encontrado: classificação=82, categoria=2723 ('Café (em grão) Total')
[OK] Códigos de variáveis resolvidos: {'area_colhida': 216, 'quantidade_produzida': 214, 'rendimento_medio': 112}

[INFO] Buscando dados de Arceburgo (3104106)...

[INFO] Buscando dados de Cabo Verde (3109501)...

[INFO] Buscando dados de Guaranésia (3128303)...

[INFO] Buscando dados de Guaxupé (3128709)...

[INFO] Buscando dados de Itamogi (3132909)...

[INFO] Buscando dados de Jacuí (3134806)...

[INFO] Buscando dados de Juruaia (3136900)...

[INFO] Buscando dados de Monte Belo (3143005)...

[INFO] Buscando dados de Muzambinho (3144102)...

[INFO] Buscando dados de Nova Resende (3145109)...

[SUCESSO] Dados salvos em: TCC_SIDRA_Historico_1988_2012.csv
Linhas obtidas: 250
Municipio  Ano  Area_Colhida_ha  Producao_t  Rendimento_kg_ha
Arceburgo

In [ ]:
import pandas as pd

def mesclar_bases():
    print("Carregando as duas bases de dados...")

    try:
        # Lê o CSV histórico que você acabou de gerar
        df_historico = pd.read_csv("TCC_SIDRA_Historico_1988_2012.csv")

        # Lê a sua planilha principal atual
        df_atual = pd.read_excel("TCC_Matriz_Fenologica_final.xlsx")

        print(f"Linhas na base histórica: {len(df_historico)}")
        print(f"Linhas na base atual: {len(df_atual)}")

        # Empilha (concatena) as duas bases. O Pandas automaticamente coloca NaN
        # nas colunas de Clima/NDVI para os anos de 1988 a 2012.
        df_unificado = pd.concat([df_historico, df_atual], ignore_index=True)

        # Organiza tudo por Município e Ordem Cronológica
        df_unificado = df_unificado.sort_values(by=["Municipio", "Ano"]).reset_index(drop=True)

        # Salva o arquivo final
        nome_saida = "TCC_Matriz_Unificada_1988_2023.xlsx"
        df_unificado.to_excel(nome_saida, index=False)

        print("\n==================================================")
        print(f"✅ SUCESSO! Base unificada salva como: {nome_saida}")
        print(f"Total de linhas agora: {len(df_unificado)}")
        print("==================================================")

    except FileNotFoundError as e:
        print(f"\n[ERRO] Arquivo não encontrado: {e}")
        print("Certifique-se de que o CSV histórico e a planilha Suprema estão na mesma pasta.")

if __name__ == "__main__":
    mesclar_bases()

Carregando as duas bases de dados...
Linhas na base histórica: 250
Linhas na base atual: 140

✅ SUCESSO! Base unificada salva como: TCC_Matriz_Unificada_1988_2023.xlsx
Total de linhas agora: 390


In [ ]:
EXTRAÇÃO DO CLIMA HISTÓRICO

In [ ]:
import requests
import pandas as pd
import time
import numpy as np

def extrair_clima_historico():
    print("==================================================")
    print("🌍 INICIANDO EXTRAÇÃO CLIMÁTICA HISTÓRICA (ERA5)")
    print("==================================================")

    # Coordenadas geográficas das 14 cidades da microrregião de SSP
    # Isso evita o uso de APIs de geocodificação que podem falhar ou bloquear o IP
    cidades_coords = {
        'São Sebastião do Paraíso': (-20.9172, -46.9906),
        'Muzambinho': (-21.3755, -46.5277),
        'Guaxupé': (-21.3052, -46.7122),
        'Cabo Verde': (-21.4727, -46.3980),
        'Nova Resende': (-21.1238, -46.4216),
        'Arceburgo': (-21.3636, -46.9391),
        'Bom Jesus da Penha': (-21.0152, -46.5177),
        'Capetinga': (-20.6127, -47.0083),
        'Itamogi': (-21.2369, -47.0177),
        'Jacuí': (-21.0583, -46.6694),
        'Juruaia': (-21.3197, -46.6027),
        'Monte Santo de Minas': (-21.1963, -46.9805),
        'São Pedro da União': (-21.1352, -46.6122),
        'São Tomás de Aquino': (-20.7852, -47.1008)
    }

    # Período histórico para casar com a nova base do IBGE
    data_inicio = "1988-01-01"
    data_fim = "2012-12-31"

    resultados_anuais = []

    for cidade, (lat, lon) in cidades_coords.items():
        print(f"[{cidade}] Baixando mais de 9.000 dias de dados climáticos...")

        # URL da API Open-Meteo (Archive) - Gratuita e sem chave de acesso
        # Parâmetros: temp mínima a 2 metros de altura e soma de chuva diária
        url = (
            f"https://archive-api.open-meteo.com/v1/archive?"
            f"latitude={lat}&longitude={lon}&"
            f"start_date={data_inicio}&end_date={data_fim}&"
            f"daily=temperature_2m_min,precipitation_sum&"
            f"timezone=America%2FSao_Paulo"
        )

        try:
            resposta = requests.get(url, timeout=30)
            resposta.raise_for_status()
            dados = resposta.json()

            # Monta um DataFrame com os dados diários
            df_diario = pd.DataFrame({
                'Data': pd.to_datetime(dados['daily']['time']),
                'Tmin': dados['daily']['temperature_2m_min'],
                'Precip': dados['daily']['precipitation_sum']
            })

            # Extrai o Ano da data
            df_diario['Ano'] = df_diario['Data'].dt.year

            # Agrupa os 365 dias para criar as métricas anuais exigidas pela nossa IA
            df_anual = df_diario.groupby('Ano').agg(
                Tmin_Abs=('Tmin', 'min'),                         # A menor temperatura registrada no ano
                Dias_Abaixo_3C=('Tmin', lambda x: (x < 3.0).sum()), # Dias com risco de geada
                Precip_Anual=('Precip', 'sum')                    # Chuva total acumulada
            ).reset_index()

            # Adiciona o nome do município
            df_anual['Municipio'] = cidade

            resultados_anuais.append(df_anual)

            # Pausa de 1 segundo para não sobrecarregar e ser banido do servidor
            time.sleep(1)

        except Exception as e:
            print(f"❌ [ERRO] Falha ao processar {cidade}: {e}")

    if resultados_anuais:
        # Junta todas as cidades
        df_final = pd.concat(resultados_anuais, ignore_index=True)

        # Ordena as colunas para o padrão do nosso TCC
        df_final = df_final[['Ano', 'Municipio', 'Tmin_Abs', 'Precip_Anual', 'Dias_Abaixo_3C']]

        # Ordena as linhas por Cidade e Ano
        df_final = df_final.sort_values(by=['Municipio', 'Ano']).reset_index(drop=True)

        # Salva o arquivo Excel
        nome_arquivo = 'Clima_Historico_ERA5_1988_2012.xlsx'
        df_final.to_excel(nome_arquivo, index=False)

        print("\n==================================================")
        print(f"✅ SUCESSO! Base climática histórica salva: {nome_arquivo}")
        print(f"Foram analisados {len(df_final)} anos (linhas) para as cidades.")
        print("==================================================")
        print(df_final.head(10).to_string(index=False))
    else:
        print("\n❌ [ERRO FATAL] Nenhum dado foi baixado.")

if __name__ == "__main__":
    extrair_clima_historico()

🌍 INICIANDO EXTRAÇÃO CLIMÁTICA HISTÓRICA (ERA5)
[São Sebastião do Paraíso] Baixando mais de 9.000 dias de dados climáticos...
[Muzambinho] Baixando mais de 9.000 dias de dados climáticos...
[Guaxupé] Baixando mais de 9.000 dias de dados climáticos...
[Cabo Verde] Baixando mais de 9.000 dias de dados climáticos...
[Nova Resende] Baixando mais de 9.000 dias de dados climáticos...
[Arceburgo] Baixando mais de 9.000 dias de dados climáticos...
❌ [ERRO] Falha ao processar Arceburgo: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=-21.3636&longitude=-46.9391&start_date=1988-01-01&end_date=2012-12-31&daily=temperature_2m_min,precipitation_sum&timezone=America%2FSao_Paulo
[Bom Jesus da Penha] Baixando mais de 9.000 dias de dados climáticos...
❌ [ERRO] Falha ao processar Bom Jesus da Penha: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=-21.0152&longitude=-46.5177&start_date=1988-01-01&end_d

MESCLANDO AS TABELAS


In [ ]:
"""
=====================================================================
TCC - Empilhamento Final das Matrizes de Dados (1988-2023)
=====================================================================

Este script automatiza a união das 3 peças do seu trabalho:
1. Sua matriz atual (TCC_Matriz_Fenologica_SUPREMA) com dados de 2013-2023.
2. Os dados de produtividade do IBGE (1988-2012).
3. Os dados de clima do ERA5/Open-Meteo (1988-2012).

O resultado será a 'TCC_Matriz_Unificada_1988_2023.xlsx' perfeita
para rodar no pipeline analítico final.
"""

import pandas as pd
import numpy as np
import os

def unificar_matrizes_tcc():
    print("==================================================")
    print("🚀 INICIANDO UNIFICAÇÃO DA MATRIZ DO TCC")
    print("==================================================")

    # Nomes dos arquivos
    arq_matriz_atual = 'TCC_Matriz_Fenologica_final.xlsx'
    arq_ibge_hist = 'TCC_SIDRA_Historico_1988_2012.csv'
    arq_clima_hist = 'Clima_Historico_ERA5_1988_2012.xlsx'

    # Verifica se os arquivos existem antes de tentar ler
    arquivos_necessarios = [arq_matriz_atual, arq_ibge_hist, arq_clima_hist]
    arquivos_faltantes = [arq for arq in arquivos_necessarios if not os.path.exists(arq)]

    if arquivos_faltantes:
        print("❌ [ERRO] Os seguintes arquivos não foram encontrados na pasta:")
        for arq in arquivos_faltantes:
            print(f"   - {arq}")
        print("\nPor favor, certifique-se de que os 3 arquivos estão no mesmo diretório deste script.")
        return

    try:
        print("1. Carregando dados...")
        df_atual = pd.read_excel(arq_matriz_atual)
        df_ibge_hist = pd.read_csv(arq_ibge_hist)
        df_clima_hist = pd.read_excel(arq_clima_hist)

        print("2. Cruzando Produtividade Histórica (IBGE) com Clima Histórico (ERA5)...")
        # Junta o clima com a produção usando Município e Ano como chave
        df_historico_completo = pd.merge(
            df_ibge_hist,
            df_clima_hist,
            on=['Municipio', 'Ano'],
            how='inner' # Só mantém se tiver as duas informações no ano/cidade
        )

        print("3. Preenchendo colunas de NDVI antigas com NaN (vazio)...")
        # Como não tinha Sentinel-2 entre 1988 e 2012, essas colunas ficam vazias
        colunas_ndvi = ['NDVI_Vigor_Maximo', 'NDVI_Estresse_Maximo', 'NDVI_Florada', 'NDVI_Amplitude']
        for col in colunas_ndvi:
            # Só cria a coluna se ela já existir na matriz Suprema
            if col in df_atual.columns:
                df_historico_completo[col] = np.nan

        # Garante que todas as colunas da matriz histórica existam na matriz atual e vice-versa
        # Isso evita erros de concatenação
        colunas_faltantes_hist = set(df_atual.columns) - set(df_historico_completo.columns)
        for col in colunas_faltantes_hist:
            df_historico_completo[col] = np.nan

        colunas_faltantes_atual = set(df_historico_completo.columns) - set(df_atual.columns)
        for col in colunas_faltantes_atual:
             df_atual[col] = np.nan

        # Reordena as colunas para baterem
        df_historico_completo = df_historico_completo[df_atual.columns]

        print("4. Empilhando a base Histórica (1988-2012) com a Base Atual (2013-2023)...")
        df_unificado = pd.concat([df_historico_completo, df_atual], ignore_index=True)

        print("5. Organizando chronologicamente por Município...")
        df_unificado = df_unificado.sort_values(by=['Municipio', 'Ano']).reset_index(drop=True)

        nome_saida = 'TCC_Matriz_Unificada_1988_2023.xlsx'
        df_unificado.to_excel(nome_saida, index=False)

        print("\n==================================================")
        print(f"✅ SUCESSO ABSOLUTO! Matriz unificada e salva como: {nome_saida}")
        print(f"Total de Anos x Municípios (Linhas): {len(df_unificado)}")
        print(f"Período coberto: {df_unificado['Ano'].min()} a {df_unificado['Ano'].max()}")
        print("==================================================")

    except Exception as e:
         print(f"❌ [ERRO INESPERADO]: {e}")

if __name__ == "__main__":
    unificar_matrizes_tcc()

🚀 INICIANDO UNIFICAÇÃO DA MATRIZ DO TCC
❌ [ERRO] Os seguintes arquivos não foram encontrados na pasta:
   - TCC_Matriz_Fenologica_final.xlsx
   - TCC_SIDRA_Historico_1988_2012.csv
   - Clima_Historico_ERA5_1988_2012.xlsx

Por favor, certifique-se de que os 3 arquivos estão no mesmo diretório deste script.
